<a href="https://colab.research.google.com/github/kevinn78/salud_mental/blob/main/Proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [113]:
# --- FASE 1: DIAGNÓSTICO ESPECÍFICO ---
import pandas as pd
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
path_raw = '/content/drive/MyDrive/semestre 5/PROGRAMACION PARA LA CIENCIA DE DATOS/data/raw/'
df_mental = pd.read_csv(os.path.join(path_raw, 'Dataset_salud_mental.zip'), compression='zip', encoding='latin-1')

print("🚨 REPORTE DE INCONSISTENCIAS GEOGRÁFICAS:")
# Mostramos registros donde el Country Code está vacío o tiene signos de pregunta
display(df_mental[['Country', 'Country_Code']].head(20))


Mounted at /content/drive
🚨 REPORTE DE INCONSISTENCIAS GEOGRÁFICAS:


,Country,Country_Code
0,NaN,BR
1,Argentina,ES
2,BRASIL,CL
3,Peru,AR
4,Peru,PE
5,Chile,US
6,Spain,NaN
7,BRASIL,ES
8,NaN,BR
9,Argentina,??


In [121]:
# --- FASE 2:  ---
import numpy as np

print("🛠️ INICIANDO TRATAMIENTO ARTESANAL...\n")
df_clean = df_mental.copy()

# --- PASO 1: TRATAMIENTO DE NULOS (NaN) ---
df_clean.dropna(subset=['Country'], inplace=True)
print("1️⃣ [PASO NULOS]: Registros vacíos eliminados.")
display(df_clean[['Country', 'Country_Code']].head(5))

# --- PASO 2: MAPEO MANUAL (PAÍSES Y CÓDIGOS) ---
# Aquí corregimos Argentina, Perú, España y EE.UU. de forma manual
def mapeo_artesanal(row):
    pais = str(row['Country']).strip().upper()
    if 'ARGENT' in pais:
        row['Country'], row['Country_Code'] = 'Argentina', 'AR'
    elif 'PERU' in pais:
        row['Country'], row['Country_Code'] = 'Peru', 'PE'
    elif 'SPAIN' in pais or 'ESPA' in pais:
        row['Country'], row['Country_Code'] = 'España', 'ES'
    elif any(x in pais for x in ['USA', 'UNITED STATES', 'EEUU']):
        row['Country'], row['Country_Code'] = 'Estados Unidos', 'US'
    elif 'CHILE' in pais:
        row['Country'], row['Country_Code'] = 'Chile', 'CL'
    elif 'BRASIL' in pais or 'BRAZIL' in pais:
        row['Country'], row['Country_Code'] = 'Brasil', 'BR'
    else:
        row['Country'] = pais.title()
    return row

df_clean = df_clean.apply(mapeo_artesanal, axis=1)
print("\n2️⃣ [PASO MAPEO]: Nombres y códigos sincronizados manualmente.")
display(df_clean[['Country', 'Country_Code']].head(5))

# --- PASO 3: TRATAMIENTO DE DUPLICADOS ---
df_clean.drop_duplicates(inplace=True)
print("\n3️⃣ [PASO DUPLICADOS]: Filas repetidas eliminadas.")
display(df_clean[['Country', 'Country_Code']].head(5))

# --- PASO 4: CONTROL DE VALORES ATÍPICOS (OUTLIERS) ---
# Decidimos un rango de 2 a 14 horas.
# Mantenemos los extremos (2 y 14) porque en salud mental, dormir muy poco o demasiado
# son indicadores críticos de riesgo. Solo eliminamos lo biológicamente imposible.


df_clean['sleep_hours'] = np.clip(df_clean['sleep_hours'], a_min=2, a_max=14)
print("\n4️⃣ [PASO OUTLIERS]: Horas de sueño normalizadas (2-14 hrs).")
df_clean.reset_index(drop=True, inplace=True)
display(df_clean[['Country', 'Country_Code', 'sleep_hours']].head(5))

🛠️ INICIANDO TRATAMIENTO ARTESANAL...

1️⃣ [PASO NULOS]: Registros vacíos eliminados.


,Country,Country_Code
1,Argentina,ES
2,BRASIL,CL
3,Peru,AR
4,Peru,PE
5,Chile,US



2️⃣ [PASO MAPEO]: Nombres y códigos sincronizados manualmente.


,Country,Country_Code
1,Argentina,AR
2,Brasil,BR
3,Peru,PE
4,Peru,PE
5,Chile,CL



3️⃣ [PASO DUPLICADOS]: Filas repetidas eliminadas.


,Country,Country_Code
1,Argentina,AR
2,Brasil,BR
3,Peru,PE
4,Peru,PE
5,Chile,CL



4️⃣ [PASO OUTLIERS]: Horas de sueño normalizadas (2-14 hrs).


,Country,Country_Code,sleep_hours
0,Argentina,AR,6.2
1,Brasil,BR,5.3
2,Peru,PE,7.1
3,Peru,PE,5.9
4,Chile,CL,7.5
